# Unified LLM Agent Evaluation Pipeline

**End-to-end walkthrough of every utility function in `utility.py`.**

| Section | Covers |
|---------|--------|
| 1 | Imports & setup |
| 2 | Trace creation & step logging |
| 3 | Persistence — JSONL & SQLite |
| 4 | RAGAS-style grounding metrics |
| 5 | DeepEval-style correctness metrics |
| 6 | Custom heuristics — tool efficiency & redundancy |
| 7 | Cost & latency tracking |
| 8 | Unified evaluation layer |
| 9 | Reliability scoring & grading |
| 10 | Cross-run comparison & leaderboard |
| 11 | Reporting & export |
| 12 | Full end-to-end demo (multi-run) |


---
## 1  Imports & Setup

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────
import json, time, pprint
from pathlib import Path

# ── All utility functions ─────────────────────────────────────────────────
from utility import (
    # Constants / types
    COST_PER_INPUT_TOKEN, COST_PER_OUTPUT_TOKEN, RELIABILITY_WEIGHTS,
    TraceRecord, EvalResult, Timer,

    # 2 — Trace logging
    generate_run_id, create_trace, log_step, log_tool_call, finalise_trace,

    # 3 — Persistence
    save_trace_jsonl, load_traces_jsonl,
    init_sqlite, save_trace_sqlite, load_traces_sqlite,
    save_eval_result_sqlite,

    # 4 — RAGAS grounding
    compute_faithfulness, compute_context_relevance,
    compute_grounding_score, run_ragas_evaluation,

    # 5 — DeepEval correctness
    compute_answer_relevance, compute_correctness,
    compute_hallucination_risk, run_deepeval_evaluation,

    # 6 — Custom heuristics
    compute_tool_efficiency, compute_redundancy_ratio,
    detect_redundant_calls, run_custom_heuristics,

    # 7 — Cost & latency
    compute_cost, cost_to_score,
    compute_per_step_latency, latency_to_score,
    run_cost_latency_analysis,

    # 8 — Unified evaluation
    evaluate_trace,

    # 9 — Reliability scoring
    compute_reliability_score, reliability_grade,

    # 10 — Cross-run comparison
    compare_runs, filter_runs_by_task, top_k_runs,

    # 11 — Reporting
    format_eval_summary, export_results_json, load_results_json,
)

print("✅  All utility functions imported successfully.")
print(f"   Cost rates  → input: ${COST_PER_INPUT_TOKEN}/tok  output: ${COST_PER_OUTPUT_TOKEN}/tok")
print(f"   Reliability weights → {RELIABILITY_WEIGHTS}")

---
## 2  Trace Creation & Step Logging

Functions: `generate_run_id`, `create_trace`, `log_step`, `log_tool_call`, `finalise_trace`, `Timer`

In [ ]:
# ── 2-A  Generate a unique run ID ────────────────────────────────────────
run_id = generate_run_id()
print("New run_id:", run_id)

In [ ]:
# ── 2-B  Create a trace ───────────────────────────────────────────────────
TASK_ID    = "revenue_query"
TASK_INPUT = "What was the total revenue for Q3 2024 across all product lines?"

trace = create_trace(task_id=TASK_ID, task_input=TASK_INPUT, run_id=run_id)
print("Trace created:")
print(f"  run_id  : {trace.run_id}")
print(f"  task_id : {trace.task_id}")
print(f"  input   : {trace.task_input}")

In [ ]:
# ── 2-C  Log reasoning steps with the Timer context manager ──────────────

with Timer() as t:
    time.sleep(0.05)   # simulate LLM thinking
step1 = log_step(
    trace,
    step_type    = "reasoning",
    content      = "I need to retrieve Q3 2024 revenue data from the financial database.",
    input_tokens = 120,
    output_tokens= 45,
    latency_ms   = t.elapsed_ms,
)
print("Step 1 logged:", step1)

In [ ]:
# ── 2-D  Log a RAG tool call ─────────────────────────────────────────────
CONTEXTS = [
    "Q3 2024 Product A revenue: $4.2M. Product B revenue: $3.1M.",
    "Total Q3 2024 revenue across all product lines was $12.7 million, a 15% YoY increase.",
    "Q3 2024 Product C revenue: $5.4M.",
]

with Timer() as t:
    time.sleep(0.03)
rag_call = log_tool_call(
    trace,
    tool_name  = "rag_retrieval",
    tool_input = {"query": TASK_INPUT, "top_k": 3},
    tool_output= CONTEXTS,
    latency_ms = t.elapsed_ms,
    success    = True,
)
print("RAG tool call logged:", rag_call["tool_name"], "→ success:", rag_call["success"])

In [ ]:
# ── 2-E  Log a Python-exec tool call ─────────────────────────────────────
with Timer() as t:
    time.sleep(0.01)
py_call = log_tool_call(
    trace,
    tool_name  = "python_exec",
    tool_input = {"code": "4.2 + 3.1 + 5.4"},
    tool_output= 12.7,
    latency_ms = t.elapsed_ms,
    success    = True,
)

# Simulate a duplicate (redundant) call
dup_call = log_tool_call(
    trace,
    tool_name  = "python_exec",
    tool_input = {"code": "4.2 + 3.1 + 5.4"},   # identical input → redundant
    tool_output= 12.7,
    latency_ms = t.elapsed_ms,
    success    = True,
)
print(f"Total tool calls so far: {len(trace.tool_calls)}")

In [ ]:
# ── 2-F  Finalise the trace with the agent's final response ──────────────
AGENT_ANSWER    = "The total revenue for Q3 2024 across all product lines was $12.7 million."
GROUND_TRUTH    = "Total Q3 2024 revenue was $12.7 million, representing a 15% year-over-year increase."

with Timer() as t:
    time.sleep(0.02)
log_step(trace, "reasoning", "Summing product line revenues confirms $12.7M total.",
         input_tokens=200, output_tokens=60, latency_ms=t.elapsed_ms)

trace = finalise_trace(trace, final_response=AGENT_ANSWER)

print("Trace finalised.")
print(f"  Steps       : {len(trace.steps)}")
print(f"  Tool calls  : {len(trace.tool_calls)}")
print(f"  Total tokens: {trace.total_input_tokens}in / {trace.total_output_tokens}out")
print(f"  Total latency: {trace.total_latency_ms:.1f} ms")

---
## 3  Persistence — JSONL & SQLite

Functions: `save_trace_jsonl`, `load_traces_jsonl`, `init_sqlite`, `save_trace_sqlite`, `load_traces_sqlite`, `save_eval_result_sqlite`

In [ ]:
# ── 3-A  JSONL persistence ────────────────────────────────────────────────
JSONL_PATH = "/tmp/traces.jsonl"

save_trace_jsonl(trace, path=JSONL_PATH)
loaded_traces = load_traces_jsonl(path=JSONL_PATH)

print(f"Saved and reloaded {len(loaded_traces)} trace(s) from JSONL.")
print("First trace run_id:", loaded_traces[0]["run_id"])

In [ ]:
# ── 3-B  SQLite persistence ───────────────────────────────────────────────
DB_PATH = "/tmp/traces.db"
conn    = init_sqlite(db_path=DB_PATH)

save_trace_sqlite(trace, conn)

all_db_traces = load_traces_sqlite(conn)
task_traces   = load_traces_sqlite(conn, task_id=TASK_ID)

print(f"SQLite — total traces : {len(all_db_traces)}")
print(f"SQLite — task '{TASK_ID}': {len(task_traces)} trace(s)")

---
## 4  RAGAS-Style Grounding Metrics

Functions: `compute_faithfulness`, `compute_context_relevance`, `compute_grounding_score`, `run_ragas_evaluation`

In [ ]:
# ── 4-A  Individual grounding metrics ────────────────────────────────────
faithfulness = compute_faithfulness(AGENT_ANSWER, CONTEXTS)
ctx_relevance = compute_context_relevance(TASK_INPUT, CONTEXTS)
grounding    = compute_grounding_score(faithfulness, ctx_relevance)

print(f"Faithfulness      : {faithfulness:.4f}")
print(f"Context Relevance : {ctx_relevance:.4f}")
print(f"Grounding Score   : {grounding:.4f}")

In [ ]:
# ── 4-B  Full RAGAS evaluation bundle ────────────────────────────────────
ragas_metrics = run_ragas_evaluation(
    query              = TASK_INPUT,
    answer             = AGENT_ANSWER,
    retrieved_contexts = CONTEXTS,
)
print("RAGAS metrics:")
pprint.pprint(ragas_metrics)

---
## 5  DeepEval-Style Correctness Metrics

Functions: `compute_answer_relevance`, `compute_correctness`, `compute_hallucination_risk`, `run_deepeval_evaluation`

In [ ]:
# ── 5-A  Individual correctness metrics ──────────────────────────────────
ans_relevance    = compute_answer_relevance(AGENT_ANSWER, TASK_INPUT)
correctness_f1   = compute_correctness(AGENT_ANSWER, GROUND_TRUTH, method="token_f1")
correctness_exact= compute_correctness(AGENT_ANSWER, GROUND_TRUTH, method="exact_match")
hall_risk        = compute_hallucination_risk(AGENT_ANSWER, CONTEXTS, GROUND_TRUTH)

print(f"Answer Relevance      : {ans_relevance:.4f}")
print(f"Correctness (Token F1): {correctness_f1:.4f}")
print(f"Correctness (Exact)   : {correctness_exact:.4f}")
print(f"Hallucination Risk    : {hall_risk:.4f}")

In [ ]:
# ── 5-B  Full DeepEval bundle ─────────────────────────────────────────────
deepeval_metrics = run_deepeval_evaluation(
    query              = TASK_INPUT,
    answer             = AGENT_ANSWER,
    ground_truth       = GROUND_TRUTH,
    retrieved_contexts = CONTEXTS,
)
print("DeepEval metrics:")
pprint.pprint(deepeval_metrics)

---
## 6  Custom Heuristics — Tool Efficiency & Redundancy

Functions: `compute_tool_efficiency`, `compute_redundancy_ratio`, `detect_redundant_calls`, `run_custom_heuristics`

In [ ]:
# ── 6-A  Individual heuristic metrics ────────────────────────────────────
efficiency  = compute_tool_efficiency(trace.tool_calls)
redundancy  = compute_redundancy_ratio(trace.tool_calls)
redundant_i = detect_redundant_calls(trace.tool_calls)

print(f"Tool Efficiency   : {efficiency:.4f}  (lower = wasted calls)")
print(f"Redundancy Ratio  : {redundancy:.4f}  (higher = more duplicates)")
print(f"Redundant call idx: {redundant_i}")

In [ ]:
# ── 6-B  Full heuristics bundle ──────────────────────────────────────────
heuristic_metrics = run_custom_heuristics(trace.tool_calls)
print("Custom heuristic metrics:")
pprint.pprint(heuristic_metrics)

---
## 7  Cost & Latency Tracking

Functions: `compute_cost`, `cost_to_score`, `compute_per_step_latency`, `latency_to_score`, `run_cost_latency_analysis`

In [ ]:
# ── 7-A  Cost ──────────────────────────────────────────────────────────────
cost_usd   = compute_cost(trace.total_input_tokens, trace.total_output_tokens)
c_score    = cost_to_score(cost_usd, budget_usd=0.05)

print(f"Estimated cost  : ${cost_usd:.6f}")
print(f"Cost score      : {c_score:.4f}  (1.0 = well within budget)")

In [ ]:
# ── 7-B  Latency ──────────────────────────────────────────────────────────
per_step_lat = compute_per_step_latency(trace.steps)
lat_score    = latency_to_score(trace.total_latency_ms, threshold_ms=30_000)

print(f"Per-step latencies (ms): {per_step_lat}")
print(f"Total latency          : {trace.total_latency_ms:.1f} ms")
print(f"Latency score          : {lat_score:.4f}  (1.0 = very fast)")

In [ ]:
# ── 7-C  Full cost & latency bundle ──────────────────────────────────────
cl_metrics = run_cost_latency_analysis(trace)
print("Cost & latency metrics:")
pprint.pprint({k: v for k, v in cl_metrics.items() if k != "per_step_latency_ms"})
print(f"  per_step_latency_ms : {cl_metrics['per_step_latency_ms']}")

---
## 8  Unified Evaluation Layer

Function: `evaluate_trace`  
*(internally chains all RAGAS / DeepEval / heuristic / cost-latency calls)*

In [ ]:
result = evaluate_trace(
    trace              = trace,
    ground_truth       = GROUND_TRUTH,
    retrieved_contexts = CONTEXTS,
)

print("EvalResult fields:")
for attr in [
    "run_id", "task_id", "correctness", "grounding",
    "faithfulness", "context_relevance", "tool_efficiency",
    "redundancy_ratio", "hallucination_risk",
    "cost_usd", "cost_score", "latency_ms", "latency_score",
    "reliability_score",
]:
    val = getattr(result, attr)
    print(f"  {attr:<22}: {val}")

In [ ]:
# Save eval result to SQLite
save_eval_result_sqlite(result, conn)
print("EvalResult saved to SQLite.")

---
## 9  Reliability Scoring & Grading

Functions: `compute_reliability_score`, `reliability_grade`

In [ ]:
# ── 9-A  Default weights ──────────────────────────────────────────────────
r_score = compute_reliability_score(result)
grade   = reliability_grade(r_score)

print(f"Reliability score (default weights): {r_score:.4f}  →  Grade: {grade}")

In [ ]:
# ── 9-B  Custom weights (correctness-heavy) ───────────────────────────────
custom_weights = {
    "correctness":     0.50,
    "grounding":       0.20,
    "tool_efficiency": 0.10,
    "hallucination":   0.10,
    "cost_score":      0.05,
    "latency_score":   0.05,
}
r_score_custom = compute_reliability_score(result, weights=custom_weights)
print(f"Reliability score (custom weights) : {r_score_custom:.4f}  →  Grade: {reliability_grade(r_score_custom)}")

---
## 10  Cross-Run Comparison & Leaderboard

Functions: `compare_runs`, `filter_runs_by_task`, `top_k_runs`

In [ ]:
# ── 10-A  Synthesise additional mock runs for comparison ─────────────────
def make_mock_result(task_id, correctness, grounding, efficiency, hall_risk, cost, latency):
    """Helper to build mock EvalResult objects for demonstration."""
    r = EvalResult(
        run_id=generate_run_id(),
        task_id=task_id,
        correctness=correctness,
        grounding=grounding,
        faithfulness=grounding,
        context_relevance=grounding,
        tool_efficiency=efficiency,
        redundancy_ratio=1.0 - efficiency,
        hallucination_risk=hall_risk,
        cost_usd=cost,
        cost_score=cost_to_score(cost),
        latency_ms=latency,
        latency_score=latency_to_score(latency),
    )
    r.reliability_score = compute_reliability_score(r)
    return r

all_results = [
    result,   # the real run from earlier
    make_mock_result("revenue_query",  0.72, 0.68, 0.80, 0.30, 0.003, 4200),
    make_mock_result("revenue_query",  0.88, 0.81, 0.95, 0.10, 0.001, 2100),
    make_mock_result("rag_qa",         0.60, 0.55, 0.70, 0.45, 0.006, 9000),
    make_mock_result("rag_qa",         0.91, 0.87, 1.00, 0.05, 0.002, 3200),
]

print(f"Total runs in comparison pool: {len(all_results)}")

In [ ]:
# ── 10-B  Compare all runs ────────────────────────────────────────────────
comparison = compare_runs(all_results)

print(f"Runs compared: {comparison['n_runs']}\n")
print("Per-metric aggregates (mean / std / best / worst):")
for metric, stats in comparison["metrics"].items():
    print(f"  {metric:<22}: mean={stats['mean']:.3f}  std={stats['std']:.3f}  "
          f"best={stats['best']:.3f}  worst={stats['worst']:.3f}")

In [ ]:
# ── 10-C  Leaderboard ─────────────────────────────────────────────────────
print("\nLeaderboard:")
print(f"{'Rank':<6} {'Run ID':<12} {'Task':<16} {'Score':<8} Grade")
print("─" * 52)
for entry in comparison["leaderboard"]:
    print(
        f"{entry['rank']:<6} "
        f"{entry['run_id'][:8]:<12} "
        f"{entry['task_id']:<16} "
        f"{entry['reliability_score']:<8.4f} "
        f"{entry['grade']}"
    )

In [ ]:
# ── 10-D  Filter by task ──────────────────────────────────────────────────
revenue_runs = filter_runs_by_task(all_results, "revenue_query")
print(f"Runs for task 'revenue_query': {len(revenue_runs)}")

# ── 10-E  Top-K runs ──────────────────────────────────────────────────────
top3 = top_k_runs(all_results, k=3)
print("\nTop-3 runs by reliability score:")
for r in top3:
    print(f"  run_id={r.run_id[:8]}  score={r.reliability_score:.4f}  grade={reliability_grade(r.reliability_score)}")

---
## 11  Reporting & Export

Functions: `format_eval_summary`, `export_results_json`, `load_results_json`

In [ ]:
# ── 11-A  Human-readable summary for one run ──────────────────────────────
summary_str = format_eval_summary(result)
print(summary_str)

In [ ]:
# ── 11-B  Export all results to JSON ──────────────────────────────────────
JSON_OUT = "/tmp/eval_results.json"
export_results_json(all_results, path=JSON_OUT)
print(f"Exported {len(all_results)} results → {JSON_OUT}")

In [ ]:
# ── 11-C  Reload and inspect ──────────────────────────────────────────────
reloaded = load_results_json(JSON_OUT)
print(f"Reloaded {len(reloaded)} result(s).")
print("First result keys:", list(reloaded[0].keys()))

---
## 12  Full End-to-End Demo — Multi-Run Pipeline

Simulates three agent runs on different tasks and generates a consolidated comparison report.

In [ ]:
SCENARIOS = [
    {
        "task_id":    "rag_qa",
        "task_input": "Who founded the company and in what year?",
        "answer":     "The company was founded by Alice Chen in 2012.",
        "truth":      "Alice Chen founded the company in 2012.",
        "contexts":   [
            "Alice Chen founded the company in 2012 after leaving Google.",
            "The firm was incorporated in Delaware in early 2012.",
        ],
        "tool_calls_spec": [
            {"name": "rag_retrieval", "inp": "founder",   "out": "Alice Chen founded..."},
            {"name": "rag_retrieval", "inp": "year",      "out": "incorporated 2012"},
        ],
    },
    {
        "task_id":    "financial_query",
        "task_input": "What is the YoY revenue growth rate for FY2024?",
        "answer":     "Revenue grew by 22% year-over-year in FY2024.",
        "truth":      "FY2024 YoY revenue growth was 22%.",
        "contexts":   [
            "FY2024 revenue: $154M vs FY2023 revenue: $126M — a 22% increase.",
        ],
        "tool_calls_spec": [
            {"name": "financial_data", "inp": "FY2024 revenue",  "out": "$154M"},
            {"name": "financial_data", "inp": "FY2023 revenue",  "out": "$126M"},
            {"name": "python_exec",    "inp": "(154-126)/126",   "out": 0.2222},
        ],
    },
    {
        "task_id":    "plot_request",
        "task_input": "Plot monthly active users for 2024.",
        "answer":     "Here is the MAU chart for 2024 showing growth from 1.2M to 2.8M users.",
        "truth":      "MAU grew from 1.2M in Jan 2024 to 2.8M in Dec 2024.",
        "contexts":   [
            "Jan 2024 MAU: 1.2M; Jun 2024 MAU: 2.1M; Dec 2024 MAU: 2.8M.",
        ],
        "tool_calls_spec": [
            {"name": "financial_data", "inp": "MAU 2024",        "out": [1.2, 2.1, 2.8]},
            {"name": "plotting",       "inp": "line chart MAU",  "out": "chart.png"},
        ],
    },
]

e2e_results: list[EvalResult] = []

for scenario in SCENARIOS:
    # Build trace
    tr = create_trace(scenario["task_id"], scenario["task_input"])

    with Timer() as t:
        time.sleep(0.02)
    log_step(tr, "reasoning", f"Planning to answer: {scenario['task_input']}",
             input_tokens=100, output_tokens=40, latency_ms=t.elapsed_ms)

    for tc_spec in scenario["tool_calls_spec"]:
        with Timer() as t:
            time.sleep(0.01)
        log_tool_call(tr, tc_spec["name"], tc_spec["inp"], tc_spec["out"],
                      latency_ms=t.elapsed_ms, success=True)

    with Timer() as t:
        time.sleep(0.02)
    log_step(tr, "reasoning", "Synthesising final answer.",
             input_tokens=180, output_tokens=55, latency_ms=t.elapsed_ms)

    finalise_trace(tr, final_response=scenario["answer"])

    # Persist
    save_trace_jsonl(tr, path=JSONL_PATH)
    save_trace_sqlite(tr, conn)

    # Evaluate
    ev = evaluate_trace(tr, scenario["truth"], scenario["contexts"])
    save_eval_result_sqlite(ev, conn)
    e2e_results.append(ev)

    print(format_eval_summary(ev))
    print()

In [ ]:
# ── Final cross-run comparison across all e2e runs ─────────────────────
e2e_comparison = compare_runs(e2e_results)

print(f"E2E pipeline — {e2e_comparison['n_runs']} runs evaluated\n")
print("Leaderboard:")
print(f"{'Rank':<6} {'Run ID':<12} {'Task':<18} {'Score':<8} Grade")
print("─" * 55)
for entry in e2e_comparison["leaderboard"]:
    print(
        f"{entry['rank']:<6} "
        f"{entry['run_id'][:8]:<12} "
        f"{entry['task_id']:<18} "
        f"{entry['reliability_score']:<8.4f} "
        f"{entry['grade']}"
    )

print("\nReliability score summary:")
rs = e2e_comparison["metrics"]["reliability_score"]
print(f"  mean={rs['mean']}  std={rs['std']}  best={rs['best']}  worst={rs['worst']}")

export_results_json(e2e_results, "/tmp/e2e_eval_results.json")
print("\n✅  All utility functions exercised. Results exported to /tmp/e2e_eval_results.json")